
# 🧠 Analyse Complète du Dataset : Mental Health Text Classification  
## Projet : Santé Mentale & Réseaux Sociaux

---

### Source
Dataset de classification de textes sur la santé mentale.

### Objectif
Déterminer si ce dataset est adapté pour un projet de classification de troubles mentaux à partir de textes issus des réseaux sociaux.

### Pourquoi cette analyse ?
Avant de choisir un dataset final, il est important d’évaluer :

- sa qualité  
- sa structure  
- son équilibre  
- sa pertinence métier  
- sa compatibilité avec le Machine Learning / Deep Learning

---


---
# 📊 Partie 1 : Exploration et Analyse des Données
---

## 1. Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from wordcloud import WordCloud

plt.style.use('ggplot')

Nous préparons ici tous les outils nécessaires pour explorer les données, créer des visualisations plus attractives et effectuer le prétraitement NLP.

## 2. Chargement du dataset

In [ ]:
df = pd.read_csv(r"data.csv")
df

## 3. Dimensions du dataset

In [ ]:
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])

## 4. Variables disponibles

In [ ]:
print(df.columns)

## 5. Informations générales

In [ ]:
df.info()

On vérifie ici les types de données et la présence éventuelle de valeurs manquantes. Si toutes les colonnes texte sont complètes, cela réduit considérablement le travail de nettoyage.

## 6. Valeurs manquantes

In [ ]:
df.isnull().sum()

## supression des valeur null

In [ ]:
df = df.dropna()

df.isnull().sum()

## 7. Analyse des doublons

In [ ]:
print("Nombre de doublons :", df.duplicated().sum())

## 8. Distribution des classes

In [ ]:
df['status'].value_counts()

Cette étape est extrêmement importante. Elle permet de voir si certaines classes sont surreprésentées. Un dataset déséquilibré peut produire un modèle biaisé.

## 9. Visualisation des classes

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(y='status', data=df, palette='viridis')
plt.title("Distribution des classes de santé mentale")
plt.show()

Cette visualisation montre clairement quelles pathologies sont dominantes. Si certaines classes sont trop faibles, il faudra envisager du rééquilibrage.

## 10. Longueur des textes

In [ ]:
df['longueur_texte'] = df['statement'].astype(str).apply(lambda x: len(x.split()))
df['longueur_texte'].describe()

## 11. Histogramme longueur

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['longueur_texte'], bins=50, color='purple')
plt.title("Distribution de la longueur des textes")
plt.show()

In [ ]:
# distrubution du longueur du text par rapport a la classe

In [ ]:

df['word_count'] = df['statement'].apply(lambda x: len(x.split()))



avg_word_count = df.groupby('status')['word_count'].mean().sort_values()

print("Average word count per class:\n", avg_word_count)



plt.figure()
avg_word_count.plot(kind='bar')

plt.title('Average Word Count per Class')
plt.xlabel('Status')
plt.ylabel('Average Word Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



plt.figure()
df.boxplot(column='word_count', by='status')

plt.title('Word Count Distribution per Class')
plt.suptitle('')  # remove default pandas title
plt.xlabel('Status')
plt.ylabel('Word Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Une distribution relativement homogène est préférable pour entraîner des modèles NLP plus stables.

## 13. Mots fréquents

In [ ]:
all_words = " ".join(df['statement'])
freq = Counter(all_words.split())

common_words = pd.DataFrame(freq.most_common(20), columns=['Mot','Fréquence'])
common_words

Les mots les plus fréquents permettent d’identifier les thèmes dominants : stress, anxiété, dépression, etc.

## 14. Visualisation mots fréquents

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(x='Fréquence', y='Mot', data=common_words, palette='magma')
plt.title("Top 20 des mots les plus fréquents")
plt.show()

Cette visualisation donne une lecture rapide des expressions les plus présentes dans les conversations.

## 16. Tableau final d’évaluation

In [ ]:
evaluation = pd.DataFrame({
    'Critère':[
        'Taille dataset',
        'Qualité des données',
        'Labels structurés',
        'Compatibilité Machine Learning',
        'Compatibilité Deep Learning',
        'Pertinence projet santé mentale'
    ],
    'Évaluation':[
        'Très bonne',
        'Bonne',
        'Oui',
        'Excellente',
        'Très bonne',
        'Très élevée'
    ]
})
evaluation

---
# 🔠 Partie 2 : Encodage des Variables
---

In [ ]:
## ENCODAGE DE VARIABLE STATUS

In [ ]:
df["status"] = df["status"].map({"Depression": 1, "Normal": 0})

In [ ]:
df = df[df["status"].isin([0, 1])]

In [ ]:
df

---
# 🧹 Partie 3 : Prétraitement du Texte (NLP)
---

In [ ]:
#contraction expantion 

In [ ]:
import re
import contractions

def expand_contractions(text, contractions):
    words = text.split()
    new_words = []

    for word in words:
        if word in contractions:
            new_words.append(contractions[word])
        else:
            new_words.append(word)

    return " ".join(new_words)


def clean_data(text):
    text = str(text)

    # 0. normalize apostrophes
    text = text.replace("’", "'")

    # 1. lowercase
    text = text.lower()

    text = contractions.fix(text)

    # 3. replace names
    text = re.sub(r'@\w+', ' <name> ', text)

    # 4. replace numbers
    text = re.sub(r'\d+', ' <number> ', text)

    # 5. remove punctuation except ?, ! and '
    text = re.sub(r"[^\w\s?!]", "", text)

    return text


## Lemitasation 


In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer 
from nltk import pos_tag
from nltk.tokenize import word_tokenize


# download if not already done
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')



lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'  # adjective
    elif tag.startswith('V'):
        return 'v'  # verb
    elif tag.startswith('N'):
        return 'n'  # noun
    elif tag.startswith('R'):
        return 'r'  # adverb
    else:
       return 'n'


def lemmatize_tokens(statement):
    
    #tokenised statement
    statement = word_tokenize(word_tokenize)
    
    tagged = pos_tag(statement)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]


In [ ]:
df

---
# ✂️ Partie 4 : Séparation du Dataset (Train / Test)
---

In [ ]:
##separer dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['statement'],
    df['status'],
    test_size=0.2,
    random_state=42
)

In [ ]:
# creation preproccecing pipline function

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def preprocessing(X):

    X = X.copy()

    # Clean data
    X["statement"] = X["statement"].apply(clean_data)

    # Lemmatization
    X["statement"] = X["statement"].apply(lemmatize_tokens)

    return X["statement"]

# function as transformer
preprocessing_transformer = FunctionTransformer(preprocessing)

In [ ]:
#svm pipleline

In [ ]:
from sklearn.svm import SVC

# Pipeline
svm_pipeline = Pipeline([
    ('preprocessing', preprocessing_transformer),
    ('vectorization', TfidfVectorizer()),
    ('model', SVC(class_weight='balanced'))
])

# Grid search parameters
param_grid = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['linear']
}

# GridSearch
rl_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5
    n_jobs=-1
)


In [ ]:
#entraiement du model 

In [ ]:
grid.fit(X_train, y_train)


In [ ]:
#validation

In [ ]:
print(grid.best_params_)

In [ ]:
#test 

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

y_pred_svm = cross_val_predict(grid, X_test_tfidf, y_test, cv=cv)

print(classification_report(y_test, y_pred_svm))

In [ ]:
#rl pipeline

In [ ]:
from sklearn.linear_model import LogisticRegression

# Pipeline
rl_pipeline = Pipeline([
    ('preprocessing', preprocessing_transformer),
    ('vectorization', TfidfVectorizer()),
    ('model', SVC(class_weight='balanced'))
])

# Grid search parameters
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['liblinear', 'lbfgs'],
    'model__penalty': ['l2']
}

# GridSearch
rl_grid = GridSearchCV(
    estimator=rl_pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    n_jobs=-1
)


In [ ]:
#entraiment du model 

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
#validation

In [ ]:
print(grid.best_params_)

In [ ]:
#test

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

y_pred_rl = cross_val_predict(rl_grid, X_test_tfidf, y_test, cv=5)

print(classification_report(y_test, y_pred_rl))

In [ ]:
#comparaison entre les models

In [ ]:
# sauvgarder le pipline